# Module 5: Portfolio Aggregation, Risk Sensitivity & Caching Layer

This notebook demonstrates the functionality of the **Portfolio Aggregation Engine** (`src/portfolio/`) and the **Caching & Cross-Validation Layer** (`src/utils/cache.py` & `src/data/validation.py`). We will value a bond portfolio under a Nelson-Siegel spot curve, aggregate portfolio duration/convexity/DV01, compute portfolio Key Rate Durations (KRDs), simulate stress scenarios (Basel IRBB), and demonstrate local JSON caching.

In [1]:
# ruff: noqa: E402
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from src.curve.nelson_siegel import NelsonSiegelCurve
from src.instruments.bond import Bond
from src.portfolio.portfolio import Portfolio
from src.portfolio.position import Position
from src.utils.cache import JSONCache
from src.data.validation import cross_validate_yields
from src.utils.plotting import set_theme

set_theme()
print("Imports and theme loaded successfully.")

Imports and theme loaded successfully.


## 1. Construct the Base Yield Curve and Portfolio

We initialize a typical upward-sloping Nelson-Siegel spot rate curve, and build a bond portfolio consisting of two positions:
1. **Position 1**: 1,000 units of a 5-year, 4% coupon bond (face value $1,000,000).
2. **Position 2**: 500 units of a 10-year, 5% coupon bond (face value $500,000).

In [2]:
# Setup base curve: level=5%, slope=-2%, curvature=2%, decay=2.0 years
base_curve = NelsonSiegelCurve(beta0=0.05, beta1=-0.02, beta2=0.02, tau=2.0)

# Create positions
bond1 = Bond(face_value=1000.0, coupon_rate=0.04, maturity=5.0, freq=2)
pos1 = Position(bond=bond1, quantity=1000)

bond2 = Bond(face_value=1000.0, coupon_rate=0.05, maturity=10.0, freq=2)
pos2 = Position(bond=bond2, quantity=500)

# Add to portfolio
portfolio = Portfolio()
portfolio.add_position(pos1)
portfolio.add_position(pos2)

print("Portfolio initialized with 2 positions.")

Portfolio initialized with 2 positions.


## 2. Portfolio Risk Metrics & Valuation

We calculate the total market value of the portfolio, the individual position weights, and the aggregated Modified Duration, Convexity, and DV01.

In [3]:
total_mv = portfolio.market_value(base_curve)
weights = portfolio.weights(base_curve)
port_dur = portfolio.duration(base_curve)
port_conv = portfolio.convexity(base_curve)
port_dv01 = portfolio.dv01(base_curve)

print(f"Portfolio Summary:")
print(f"  Total Market Value:  ${total_mv:,.2f}")
print(f"  Modified Duration:   {port_dur:.4f} years")
print(f"  Convexity:           {port_conv:.4f}")
print(f"  DV01 (1 bp shift):   ${port_dv01:,.2f}")

df_pos = pd.DataFrame(
    {
        "Asset": ["5Y Bond (4%)", "10Y Bond (5%)"],
        "Quantity": [pos1.quantity, pos2.quantity],
        "Maturity": [pos1.bond.maturity, pos2.bond.maturity],
        "Market Value": [pos1.market_value(base_curve), pos2.market_value(base_curve)],
        "MV Weight": weights,
        "Duration": [pos1.duration(base_curve), pos2.duration(base_curve)],
        "Convexity": [pos1.convexity(base_curve), pos2.convexity(base_curve)],
        "DV01": [pos1.dv01(base_curve), pos2.dv01(base_curve)],
    }
)
df_pos.style.format(
    {
        "Market Value": "${:,.2f}",
        "MV Weight": "{:.2%}",
        "Duration": "{:.4f}",
        "Convexity": "{:.4f}",
        "DV01": "${:,.2f}",
    }
).hide(axis="index")

Portfolio Summary:
  Total Market Value:  $1,463,793.71
  Modified Duration:   5.6031 years
  Convexity:           40.4810
  DV01 (1 bp shift):   $820.18


Asset,Quantity,Maturity,Market Value,MV Weight,Duration,Convexity,DV01
5Y Bond (4%),1000,5.000000,"$963,268.18",65.81%,4.4635,23.2442,$429.96
10Y Bond (5%),500,10.000000,"$500,525.53",34.19%,7.7963,73.6536,$390.23


## 3. Portfolio Key Rate Durations (KRDs)

Key Rate Durations help identify a portfolio's sensitivity to non-parallel shifts at specific tenors. The sum of a portfolio's KRDs should match its overall Modified Duration. We compute the KRDs for the portfolio over standard tenors `[0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0]` years.

In [4]:
tenors = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
port_krds = portfolio.key_rate_duration(base_curve, tenors)

df_krd = pd.DataFrame({"Key Rate Tenor": tenors, "KRD Value": port_krds})

print("Portfolio Key Rate Durations:")
print(df_krd.to_string(index=False))
print(f"\nSum of Key Rate Durations: {np.sum(port_krds):.4f} years")
print(f"Portfolio Modified Duration: {port_dur:.4f} years")

Portfolio Key Rate Durations:
 Key Rate Tenor    KRD Value
            0.5 1.057681e-02
            1.0 3.576431e-02
            2.0 7.846557e-02
            3.0 1.835919e-01
            5.0 2.875922e+00
            7.0 2.116461e-01
           10.0 2.205208e+00
           20.0 3.883294e-13
           30.0 3.883294e-13

Sum of Key Rate Durations: 5.6012 years
Portfolio Modified Duration: 5.6031 years


## 4. Basel IRBB Stress Testing

We stress test the portfolio under the six standard Basel Committee Interest Rate Risk in the Banking Book (IRBB) standardized stress scenarios: `parallel_up`, `parallel_down`, `steepener`, `flattener`, `short_rate_up`, and `short_rate_down`.

In [5]:
scenarios = [
    "parallel_up",
    "parallel_down",
    "steepener",
    "flattener",
    "short_rate_up",
    "short_rate_down",
]
stress_results = []

for sc in scenarios:
    pnl = portfolio.stress_pnl(base_curve, sc)
    pct_mv = pnl / total_mv
    stress_results.append(
        {
            "Scenario": sc.replace("_", " ").title(),
            "P&L ($)": pnl,
            "% MV Change": pct_mv * 100.0,
        }
    )

df_stress = pd.DataFrame(stress_results)
df_stress.style.format({"P&L ($)": "${:+,.2f}", "% MV Change": "{:+.3f}%"}).hide(
    axis="index"
)

Scenario,P&L ($),% MV Change
Parallel Up,"$-79,138.09",-5.406%
Parallel Down,"$+85,067.82",+5.811%
Steepener,"$+25,733.36",+1.758%
Flattener,"$-25,078.12",-1.713%
Short Rate Up,"$-151,264.19",-10.334%
Short Rate Down,"$+171,974.72",+11.749%


## 5. Local JSON Caching Layer & Crossover Validation

We demonstrate storing and retrieving fitted Nelson-Siegel curves using the local JSON caching system (`JSONCache`) and validate yield discrepancies across sources.

In [6]:
# Initialize local cache in a temp directory for demonstration
cache = JSONCache(cache_dir=".demo_cache", default_ttl=5)

curve_params = {
    "beta0": base_curve.beta0,
    "beta1": base_curve.beta1,
    "beta2": base_curve.beta2,
    "tau": base_curve.tau,
}

# Set cache entry
cache.set("us_treasury_curve_latest", curve_params)
print("Curve parameters written to cache.")

# Get cache entry
cached_data = cache.get("us_treasury_curve_latest")
print("Cached Data:", cached_data)

# Clean up demo cache directory
cache.clear()
import shutil

shutil.rmtree(".demo_cache", ignore_errors=True)
print("Demo cache cleared and removed.")

Curve parameters written to cache.
Cached Data: {'beta0': 0.05, 'beta1': -0.02, 'beta2': 0.02, 'tau': 2.0}
Demo cache cleared and removed.


### Crossover Validation Demonstration

We cross-validate yields from two sources (e.g. FRED vs Yahoo Finance) to detect alignment errors:

In [7]:
# Source A (FRED): reports in standard percent (e.g. 4.25%)
fred_yields = {"2Y": 4.25, "10Y": 4.50}

# Source B (yfinance): reports in CBOE 10x scale (e.g. 43.10 for 4.31%)
yf_yields = {"2Y": 42.60, "10Y": 45.03}

# Validate with a 5 basis point tolerance
discrepancies = cross_validate_yields(fred_yields, yf_yields, max_diff_bps=5.0)

if len(discrepancies) > 0:
    print("Validation Discrepancies Found (exceeding 5 bps):")
    for d in discrepancies:
        print(f"  Maturity: {d['maturity']}")
        print(f"    FRED Yield (Decimal):      {d['val_a']:.4f}")
        print(f"    Yahoo Yield (Decimal):     {d['val_b']:.4f}")
        print(f"    Absolute Difference (bps): {d['diff_bps']:.2f} bps")
else:
    print("All yields matched within tolerance!")

All yields matched within tolerance!
